<a href="https://colab.research.google.com/github/russianoracle/punct-nlu-colab-training/blob/main/PunctNLU_Colab_Training_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PunctNLU CUDA Training on Colab
Training Russian punctuation restoration model on GPU

**Model:** rubert-tiny2 (29.2M params) → 12-class punctuation + capitalization

**Corpus:** 1.194M Russian sentences from Tatoeba

**Hardware:** Colab GPU (T4/A100)

## Step 1: Setup & Install Dependencies

In [3]:
!pip install -q torch transformers numpy pandas scikit-learn tqdm

In [4]:
# ── TPU Setup (Standard Installation) ──────────────────────────────────────────────────────────
# Using standard pip install as direct storage links often result in 403 Forbidden errors
!pip install -q torch-xla

try:
    import torch_xla
    import torch_xla.core.xla_model as xm
    import torch_xla.distributed.parallel_loader as pl
    print(f"✓ TPU Device detected: {xm.xla_device()}")
except Exception as e:
    print(f"⚠️ TPU not available or installation failed: {e}")
    print("Falling back to GPU/CPU logic in Config cell.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 MB 5.2 MB/s eta 0:00:00
⚠️ TPU not available or installation failed: /usr/local/lib/python3.12/dist-packages/_XLAC.cpython-312-x86_64-linux-gnu.so: undefined symbol: _ZN5torch8autograd13_wrap_outputsERKSt6vectorIN2at6TensorESaIS3_EERKSt13unordered_setIPN3c1010TensorImplESt4hashISB_ESt8equal_toISB_ESaISB_EESJ_NS9_8ArrayRefISt8optionalIS3_EEERKSt10shared_ptrINS0_4NodeEERKSt8functionIFS5_S5_S5_EESJ_RKST_IFS3_S3_EE
Falling back to GPU/CPU logic in Config cell.


In [5]:
import logging
import re
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Iterator

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModel, AutoTokenizer

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
log = logging.getLogger(__name__)

# Check GPU
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# ── Hugging Face Authentication ────────────────────────────────────────────────
try:
    from google.colab import userdata
    import os

    # Check both common variations
    token = None
    for key in ['HF_TOKEN', 'hf_token']:
        try:
            token = userdata.get(key)
            if token:
                os.environ['HF_TOKEN'] = token
                print(f"✓ {key} found and loaded from Colab Secrets")
                break
        except:
            continue

    if not token:
        print("⚠️  HF_TOKEN not detected in Secrets.")
        print("   Please ensure you added it under 🔑 Secrets AND enabled 'Notebook access'.")
except Exception as e:
    print(f"Could not access Colab Secrets: {e}")

GPU Available: True
GPU Name: Tesla T4
GPU Memory: 15.6 GB
✓ HF_TOKEN found and loaded from Colab Secrets


In [6]:
from google.colab import drive

# Optional: Mount Drive only if you need to save results there
# drive.mount('/content/drive', force_remount=True)
print("✓ Colab environment ready")

✓ Colab environment ready


In [7]:
from google.colab import drive, files
from pathlib import Path
import shutil
import os

print("=" * 80)
print("CORPUS DETECTION")
print("=" * 80)

# Check if corpus already exists in /content/
CORPUS_PATH = Path("/content/sentences.csv")
print(f"1️⃣  Checking /content/sentences.csv... {CORPUS_PATH.exists()}")

if CORPUS_PATH.exists():
    size_mb = CORPUS_PATH.stat().st_size / 1e6
    print(f"   ✓ Found: {size_mb:.0f} MB")
else:
    # Mount Google Drive as fallback
    print("2️⃣  /content not found, mounting Google Drive...")
    drive.mount('/content/drive', force_remount=True)

    # Create directory for corpus
    corpus_dir = Path("/content/drive/MyDrive/PunctNLU")
    corpus_dir.mkdir(parents=True, exist_ok=True)
    print(f"   Created: {corpus_dir}")

    CORPUS_PATH = corpus_dir / "sentences.csv"
    print(f"3️⃣  Checking Google Drive... {CORPUS_PATH.exists()}")

    # Check if corpus exists in Drive
    if CORPUS_PATH.exists():
        size_mb = CORPUS_PATH.stat().st_size / 1e6
        print(f"   ✓ Found: {size_mb:.0f} MB")
    else:
        print("4️⃣  Not in Drive either, prompting for upload...")
        print("   Click 'Choose Files' below and select sentences.csv")

        uploaded = files.upload()
        if 'sentences.csv' in uploaded:
            shutil.move('./sentences.csv', str(CORPUS_PATH))
            size_mb = CORPUS_PATH.stat().st_size / 1e6
            print(f"   ✓ Uploaded: {size_mb:.0f} MB")

print(f"\nFinal CORPUS_PATH: {CORPUS_PATH}")
print(f"Type: {type(CORPUS_PATH)}")
print(f"Exists: {CORPUS_PATH.exists()}")
print("=" * 80)

CORPUS DETECTION
1️⃣  Checking /content/sentences.csv... True
   ✓ Found: 458 MB

Final CORPUS_PATH: /content/sentences.csv
Type: <class 'pathlib.PosixPath'>
Exists: True


### 👆 Click above cell to mount Drive and upload corpus
1. Run cell above
2. If corpus not found, click "Choose Files" button
3. Select `sentences.csv` from your computer
4. Colab will upload it to Google Drive automatically (first run only ~5-10 min)
5. Run remaining cells for training!
✅ Notebook handles everything else

## Step 3: Configuration (CUDA-Optimized)

In [8]:
import os
import torch
from pathlib import Path

# ── Auto-Detect Environment ────────────────────────────────────────────────
try:
    import torch_xla.core.xla_model as xm
    IS_TPU = True
    DEVICE = xm.xla_device()
    BATCH_SIZE = 256  # TPUs handle much larger batches
    print("✨ Environment: TPU v5e-1 detected")
except:
    IS_TPU = False
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Defaulting to T4 Optimized Settings
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        print(f"✨ Environment: GPU detected ({gpu_name})")
        BATCH_SIZE = 128 # Optimized for T4 16GB VRAM
    else:
        print("✨ Environment: CPU detected")
        BATCH_SIZE = 32

# ── Persistence Paths (Google Drive) ───────────────────────────────────────
DRIVE_BASE   = Path("/content/drive/MyDrive/PunctNLU")
DRIVE_BASE.mkdir(parents=True, exist_ok=True)

CORPUS_PATH  = DRIVE_BASE / "sentences.csv"
ENCODED_PATH = DRIVE_BASE / "encoded_samples.pkl"
OUTPUT_DIR   = DRIVE_BASE / "checkpoints"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Hyperparameters ────────────────────────────────────────────────────────
MODEL_ID      = "cointegrated/rubert-tiny2"
SEQ_LEN       = 64
NUM_LABELS    = 12
NUM_EPOCHS    = 3
LEARNING_RATE = 1e-4
PUNCT_CHARS   = {",": 1, ".": 2, "!": 2, "…": 2, ";": 2, "?": 3}

print(f"✅ Configured for {DEVICE} with Batch Size: {BATCH_SIZE}")
print(f"📦 Checkpoints and Data saved to: {DRIVE_BASE}")

✨ Environment: GPU detected (Tesla T4)
✅ Configured for cuda with Batch Size: 128
📦 Checkpoints and Data saved to: /content/drive/MyDrive/PunctNLU


## Step 4: Data Loading & Processing

In [9]:
import re
import numpy as np

# Pre-compile regex for significant speedup on 1M+ rows
RE_CLEAN = re.compile(r"[^\w]", re.UNICODE)

def cap_mode_vec(words: np.ndarray) -> np.ndarray:
    modes = np.zeros(len(words), dtype=np.int32)
    for i, word in enumerate(words):
        clean = RE_CLEAN.sub("", word)
        if clean:
            alpha = [c for c in clean if c.isalpha()]
            if alpha:
                if all(c.isupper() for c in alpha): modes[i] = 2
                elif alpha[0].isupper(): modes[i] = 1
    return modes

def punct_class_vec(words: np.ndarray) -> np.ndarray:
    classes = np.zeros(len(words), dtype=np.int32)
    for i, word in enumerate(words):
        if word:
            classes[i] = PUNCT_CHARS.get(word[-1], 0)
    return classes

def word_labels_vec(words: list[str]) -> np.ndarray:
    words_arr = np.array(words, dtype=object)
    return punct_class_vec(words_arr) * 3 + cap_mode_vec(words_arr)

def ortho_to_norm(word: str) -> str:
    return RE_CLEAN.sub("", word).lower()

In [12]:
from typing import Iterator, Optional
import torch
import numpy as np
from dataclasses import dataclass

# ── Data loader ────────────────────────────────────────────────────────────

def iter_tatoeba(path: Path) -> Iterator[tuple[list[str], list[int]]]:
    """Read Tatoeba corpus without limits"""
    count = 0
    with path.open(encoding="utf-8") as f:
        for line in f:
            parts = line.rstrip("\n").split("\t", 2)
            if len(parts) != 3 or parts[1] != "rus":
                continue

            sentence = parts[2].strip()
            if len(sentence) < 5:
                continue

            words = sentence.split()
            if len(words) < 2:
                continue

            labels = word_labels_vec(words).tolist()
            norms = [ortho_to_norm(w) for w in words]

            if not any(norms):
                continue

            yield norms, labels
            count += 1
            if count % 100000 == 0:
                log.info(f"  Loaded {count} sentences...")


@dataclass
class EncodedSample:
    input_ids: torch.Tensor      # [SEQ_LEN]
    attention_mask: torch.Tensor # [SEQ_LEN]
    labels: torch.Tensor         # [SEQ_LEN]


def encode(norm_words: list[str], word_labels: list[int], tokenizer) -> Optional[EncodedSample]:
    """Encode sentence with proper word-start alignment"""
    ids = np.zeros(SEQ_LEN, dtype=np.int32)
    mask = np.zeros(SEQ_LEN, dtype=np.int32)
    labels = np.full(SEQ_LEN, -100, dtype=np.int32)

    cls_id = tokenizer.cls_token_id
    sep_id = tokenizer.sep_token_id

    ids[0] = cls_id
    mask[0] = 1
    pos = 1

    for word, lbl in zip(norm_words, word_labels):
        if pos >= SEQ_LEN - 1:
            break

        sub_ids = tokenizer.encode(word, add_special_tokens=False)
        if not sub_ids:
            sub_ids = [tokenizer.unk_token_id]

        if pos + len(sub_ids) >= SEQ_LEN:
            break

        ids[pos] = sub_ids[0]
        mask[pos] = 1
        labels[pos] = lbl
        pos += 1

        for sub_id in sub_ids[1:]:
            if pos >= SEQ_LEN:
                break
            ids[pos] = sub_id
            mask[pos] = 1
            labels[pos] = -100
            pos += 1

    if pos < SEQ_LEN:
        ids[pos] = sep_id
        mask[pos] = 1
        pos += 1

    return EncodedSample(
        input_ids=torch.from_numpy(ids).long(),
        attention_mask=torch.from_numpy(mask).long(),
        labels=torch.from_numpy(labels).long(),
    )


class PunctDataset(Dataset):
    def __init__(self, samples: list[EncodedSample]):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        return {
            "input_ids": s.input_ids,
            "attention_mask": s.attention_mask,
            "labels": s.labels,
        }

In [13]:
import multiprocessing as mp
from functools import partial
from tqdm.auto import tqdm

def encode_batch(batch, tokenizer):
    """Helper function for parallel encoding"""
    results = []
    for norm_words, labels in batch:
        sample = encode(norm_words, labels, tokenizer)
        if sample:
            results.append(sample)
    return results

def parallel_encode(corpus_data, tokenizer, num_workers=None):
    """Splits corpus into chunks and encodes them in parallel"""
    if num_workers is None:
        num_workers = mp.cpu_count()

    print(f"🚀 Parallel encoding using {num_workers} workers...")

    # Split data into chunks for workers
    chunk_size = max(1, len(corpus_data) // (num_workers * 4))
    chunks = [corpus_data[i:i + chunk_size] for i in range(0, len(corpus_data), chunk_size)]

    encode_func = partial(encode_batch, tokenizer=tokenizer)

    encoded_samples = []
    with mp.Pool(processes=num_workers) as pool:
        for result in tqdm(pool.imap_unordered(encode_func, chunks), total=len(chunks), desc="Encoding chunks"):
            encoded_samples.extend(result)

    return encoded_samples

### Updated Main with Parallel Encoding
I will now update the `main` function to use `parallel_encode` instead of the sequential loop.

In [14]:
def main_optimized():
    print("\n" + "="*50, flush=True)
    print("🚀 STARTING OPTIMIZED TRAINING PROCESS", flush=True)
    print("="*50, flush=True)

    # 1. Load corpus
    print(f"📂 Loading corpus from {CORPUS_PATH}...", flush=True)
    corpus_data = list(iter_tatoeba(CORPUS_PATH))

    # 2. Check Persistence
    encoded_samples = load_encoded_samples(str(ENCODED_PATH))

    if not encoded_samples:
        # 3. Parallel Encode
        tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
        encoded_samples = parallel_encode(corpus_data, tokenizer)
        save_encoded_samples(encoded_samples, str(ENCODED_PATH))

    # 4. Standard Pipeline
    split = int(0.95 * len(encoded_samples))
    train_dataset = PunctDataset(encoded_samples[:split])
    val_dataset = PunctDataset(encoded_samples[split:])

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, num_workers=2)

    model = PunctNLUModel(MODEL_ID).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    scheduler = torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=1.0, end_factor=0.1, total_iters=len(train_loader)*NUM_EPOCHS)
    scaler = torch.amp.GradScaler('cuda') if DEVICE.type == 'cuda' else None

    print("✅ Setup complete. Ready for training loop.")

In [15]:
import pickle

def save_encoded_samples(samples, filepath="/content/encoded_samples.pkl"):
    """Saves the list of EncodedSample objects to disk."""
    with open(filepath, 'wb') as f:
        pickle.dump(samples, f)
    print(f"✓ Successfully saved {len(samples)} samples to {filepath}")

def load_encoded_samples(filepath="/content/encoded_samples.pkl"):
    """Loads the list of EncodedSample objects from disk."""
    if Path(filepath).exists():
        with open(filepath, 'rb') as f:
            samples = pickle.load(f)
        print(f"✓ Loaded {len(samples)} samples from {filepath}")
        return samples
    else:
        print(f"✗ No saved samples found at {filepath}")
        return None

## Step 5: Model Definition

In [16]:
# ── Model ──────────────────────────────────────────────────────────────────

class PunctNLUModel(nn.Module):
    """Two-head model: punct restoration + actionable classification"""

    def __init__(self, model_id: str, num_labels: int = 12):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_id)
        self.dropout = nn.Dropout(0.1)

        # Head A1: Punctuation restoration (12 classes)
        self.punct_head = nn.Linear(self.bert.config.hidden_size, num_labels)

        # Head A2: Actionable classification (2 classes)
        self.classify_head = nn.Linear(self.bert.config.hidden_size, 2)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        hidden = outputs.last_hidden_state  # [B, L, H]
        hidden = self.dropout(hidden)

        punct_logits = self.punct_head(hidden)      # [B, L, 12]
        classify_logits = self.classify_head(hidden) # [B, L, 2]

        return punct_logits, classify_logits

## Step 6: Training Loop

In [ ]:
def train_epoch(model, loader, optimizer, scheduler, device, scaler=None):
    """FIXED version with weights for rare classes"""
    model.train()
    total_loss = 0.0
    batch_losses = []

    # Step 1: Find which classes are rare, which are common
    print("  Computing weights for rare classes...")
    all_labels = []
    for batch in loader:
        labels_batch = batch["labels"].view(-1).cpu().numpy()
        valid_labels = labels_batch[labels_batch >= 0]  # Ignore -100
        all_labels.extend(valid_labels)
    all_labels = np.array(all_labels)

    # Step 2: Count how many times each class appears
    from collections import Counter
    label_counts = Counter(all_labels)
    total_count = sum(label_counts.values())

    # Step 3: Compute weights (rare gets higher weight)
    class_weights = torch.zeros(12, device=device)
    for label_id in range(12):
        if label_id in label_counts:
            weight = total_count / (12 * label_counts[label_id])
            class_weights[label_id] = weight

    print(f"  Weights computed. Rare classes get penalty x{class_weights.max():.1f}")

    # Step 4: Create loss function with weights
    ce_loss_fn = nn.CrossEntropyLoss(weight=class_weights, ignore_index=-100)

    # Step 5: Training
    for batch_idx, batch in enumerate(loader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        device_type = 'cuda' if 'cuda' in str(device) else 'cpu'
        with torch.amp.autocast(device_type=device_type, enabled=(scaler is not None)):
            punct_logits, classify_logits = model(input_ids, attention_mask)

            # Weighted loss for punctuation
            loss_punct = ce_loss_fn(punct_logits.view(-1, 12), labels.view(-1))

            # Classification (regular loss)
            binary_labels = (labels > 0).long()
            loss_classify = nn.CrossEntropyLoss(ignore_index=-100)(
                classify_logits.view(-1, 2), binary_labels.view(-1)
            )

            # Give more attention to punctuation
            loss = 2.0 * loss_punct + 0.5 * loss_classify

        if scaler is not None:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            if 'xla' in str(device):
                xm.optimizer_step(optimizer)
            else:
                optimizer.step()

        scheduler.step()

        loss_val = loss.item()
        total_loss += loss_val
        batch_losses.append(loss_val)

        if (batch_idx + 1) % 100 == 0:
            avg_loss = np.mean(batch_losses[-100:])
            lr = optimizer.param_groups[0]['lr']
            print(f"  Batch {batch_idx + 1:4d}/{len(loader)} | loss: {loss_val:.4f} | avg_loss: {avg_loss:.4f} | LR: {lr:.2e}", flush=True)

    return total_loss / len(loader)

def eval_epoch_detailed(model, loader, device):
    """Shows the truth about model quality"""
    model.eval()
    all_predictions = []
    all_true_labels = []
    ce_loss_fn = nn.CrossEntropyLoss(ignore_index=-100)
    total_loss = 0.0

    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            logits, _ = model(input_ids, attention_mask)
            loss = ce_loss_fn(logits.view(-1, 12), labels.view(-1))
            total_loss += loss.item()

            predictions = torch.argmax(logits, dim=-1)
            all_predictions.extend(predictions[labels >= 0].cpu().numpy())
            all_true_labels.extend(labels[labels >= 0].cpu().numpy())

    all_predictions = np.array(all_predictions)
    all_true_labels = np.array(all_true_labels)
    accuracy = (all_predictions == all_true_labels).mean()

    print(f"\n📊 RESULTS:")
    print(f"   Overall accuracy: {accuracy:.1%}")
    print(f"\n   Accuracy by PUNCTUATION (main task):")

    predictions_punct = all_predictions // 3
    true_labels_punct = all_true_labels // 3

    punct_names = {0: "no punctuation", 1: "comma", 2: "period/exclamation", 3: "question"}

    for punct_type in range(4):
        mask = (true_labels_punct == punct_type)
        if mask.sum() > 0:
            correct = (predictions_punct[mask] == true_labels_punct[mask]).sum()
            total = mask.sum()
            accuracy_for_type = correct / total
            print(f"      {punct_names[punct_type]:20s}: {accuracy_for_type:>5.1%}  ({total:>6,} examples)")

    return total_loss / len(loader), accuracy

## Step 7: Main Training Function

In [ ]:
def main():
    print("\n" + "="*50, flush=True)
    print("🚀 STARTING TRAINING PROCESS", flush=True)
    print("="*50, flush=True)

    global DEVICE
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
        print(f"✓ Using Device: {torch.cuda.get_device_name(0)}", flush=True)

    # 1. Load corpus
    print(f"📂 Loading corpus from {CORPUS_PATH}...", flush=True)
    corpus_data = list(iter_tatoeba(CORPUS_PATH))
    print(f"✓ Loaded {len(corpus_data):,} sentences", flush=True)

    # 2. Load tokenizer
    print(f"🔤 Loading tokenizer: {MODEL_ID}...", flush=True)
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

    # 3. Encode samples
    print(f"⚙️  Encoding samples...", flush=True)
    encoded_samples = []
    for idx, (norm_words, labels) in enumerate(corpus_data):
        sample = encode(norm_words, labels, tokenizer)
        if sample: encoded_samples.append(sample)
        if (idx + 1) % 100000 == 0:
            print(f"  Progress: {(idx+1)/len(corpus_data)*100:.1f}% ({idx+1:,} sentences)", flush=True)

    # 4. Split train/val
    split = int(0.95 * len(encoded_samples))
    train_dataset = PunctDataset(encoded_samples[:split])
    val_dataset = PunctDataset(encoded_samples[split:])

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

    # 5. Initialize model
    print(f"🧠 Initializing model on {DEVICE}...", flush=True)
    model = PunctNLUModel(MODEL_ID).to(DEVICE)

    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    scheduler = torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=1.0, end_factor=0.1, total_iters=len(train_loader)*NUM_EPOCHS)
    scaler = torch.amp.GradScaler('cuda') if DEVICE.type == 'cuda' else None

    # 6. Training loop
    print("\n" + "="*50, flush=True)
    print("STARTING EPOCHS", flush=True)
    print("="*50, flush=True)

    best_val_loss = float("inf")
    for epoch in range(1, NUM_EPOCHS + 1):
        print(f"\n▶ Epoch {epoch}/{NUM_EPOCHS}", flush=True)
        t0 = time.time()
        train_loss = train_epoch(model, train_loader, optimizer, scheduler, DEVICE, scaler)
        val_loss, val_acc = eval_epoch_detailed(model, val_loader, DEVICE)

        print(f"  Done in {time.time()-t0:.0f}s | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.3f}", flush=True)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), OUTPUT_DIR / "best.pt")
            print(f"  ⭐ New Best Model Saved!", flush=True)

    print("\n" + "="*50, flush=True)
    print("✅ TRAINING COMPLETE", flush=True)
    print("="*50, flush=True)
    return True

In [ ]:
def main_tpu():
    print("\n" + "="*50)
    print("🚀 STARTING PERSISTENT TPU FLOW")
    print("="*50)

    device = xm.xla_device()

    # ── 1. Persistence Check (Loading from Drive) ───────────────────────────
    encoded_samples = load_encoded_samples(str(ENCODED_PATH))

    if not encoded_samples:
        print(f"📂 Loading corpus from {CORPUS_PATH}...")
        corpus_data = list(iter_tatoeba(CORPUS_PATH))

        print(f"🔤 Encoding samples (CPU process)...")
        tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
        encoded_samples = []
        for norm_words, labels in corpus_data:
            sample = encode(norm_words, labels, tokenizer)
            if sample: encoded_samples.append(sample)

        save_encoded_samples(encoded_samples, str(ENCODED_PATH))

    # ── 2. Data Pipeline ───────────────────────────────────────────────────
    split = int(0.95 * len(encoded_samples))
    train_loader = DataLoader(PunctDataset(encoded_samples[:split]), batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(PunctDataset(encoded_samples[split:]), batch_size=BATCH_SIZE)

    # ── 3. TPU Training ────────────────────────────────────────────────────
    model = PunctNLUModel(MODEL_ID).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

    # Using ParallelLoader for TPU efficiency
    print("\n🔥 Training on TPU Core...")
    for epoch in range(1, NUM_EPOCHS + 1):
        model.train()
        para_loader = pl.ParallelLoader(train_loader, [device]).per_device_loader(device)

        for batch in para_loader:
            optimizer.zero_grad()
            p_logits, c_logits = model(batch['input_ids'], batch['attention_mask'])
            # ... [Loss calculation and xm.optimizer_step(optimizer) would go here] ...
            xm.mark_step()

        # Save checkpoint to Drive after each epoch
        xm.save(model.state_dict(), OUTPUT_DIR / f"tpu_model_epoch_{epoch}.pt")
        print(f"💾 Checkpoint saved to Drive for Epoch {epoch}")

    return True

## Step 8: RUN TRAINING

In [ ]:
# ── Final Execution Logic ─────────────────────────────────────────────────────
from google.colab import drive

if not Path("/content/drive/MyDrive").exists():
    drive.mount('/content/drive')

if IS_TPU:
    print("⚡ Starting Optimized TPU Pipeline...")
    main_tpu()
else:
    print("🚀 Starting Optimized GPU Pipeline...")
    # Modify main() locally to check for persistence
    encoded_samples = load_encoded_samples(str(ENCODED_PATH))
    if encoded_samples:
        # If exists, we can skip standard main's encoding step
        print("Using persisted samples from Drive.")
    main()

In [ ]:
import cProfile
import pstats
import io

# We'll profile the main function to see the distribution of time
pr = cProfile.Profile()
pr.enable()

# Running a smaller subset or the full main to gather stats
# Note: If your dataset is huge, you might want to wrap just the encoding loop
# inside main() with these commands instead.
success = main()

pr.disable()
s = io.StringIO()
sortby = 'cumulative'  # Can also use 'tottime' to see exclusive time
ps = pstats.Stats(pr, stream=s).sort_stats(sortby)
ps.print_stats(20)  # Print the top 20 time-consuming functions
print(s.getvalue())

## Step 9: Download Results

In [ ]:
from google.colab import files
from pathlib import Path

# Check if training completed
checkpoint_dir = Path("./checkpoints/punct_nlu")
best_model = checkpoint_dir / "best.pt"
final_model = checkpoint_dir / "punct_nlu.pt"

if best_model.exists():
    print(f"Downloading trained model: {best_model.stat().st_size / 1e6:.0f} MB")
    files.download(str(best_model))
    print("✓ Downloaded: best.pt")
elif final_model.exists():
    print(f"Downloading final checkpoint: {final_model.stat().st_size / 1e6:.0f} MB")
    files.download(str(final_model))
    print("✓ Downloaded: punct_nlu.pt")
else:
    print("✗ Training output not found!")
    print(f"Checked: {checkpoint_dir}")
    print("Did training complete successfully?")
    print("Check the training log above for errors.")